# Workflow DIMOSTRAZIONE - verifica che il modello funzioni

Notebook per la **tesi**: dimostra che il modello stima correttamente,
confrontando le sue stime con la verita' nascosta dei dati sintetici.

Flusso (uguale alla produzione, ma con un passo di verifica in piu'):
1. in locale: `python -m pipeline.generator.run` crea i dati finti **e**
   `pipeline/data/ground_truth.json` (la verita' nota);
2. in locale: `app_ingestion.py` produce i 4 CSV puliti;
3. qui su Colab: carichi i 4 CSV **+ ground_truth.json**, fai il fit e poi
   il **parameter recovery** (il test).

**Prima di iniziare:** Runtime -> Cambia tipo di runtime -> GPU (T4) -> Salva.

## 1. Setup: codice e dipendenze

In [ ]:
!git clone https://github.com/Giacomod2001/Tesi-MMM.git
%cd Tesi-MMM
!pip install -q -r pipeline/requirements.txt

## 2. Carica i 4 CSV puliti + ground_truth.json

Esegui la cella, poi "Scegli file" e seleziona **cinque** file:
`media.csv`, `demand.csv`, `seasonality.csv`, `outcome.csv` (dalla cartella
`MMM_dati_puliti` sul Desktop) e `ground_truth.json` (in locale lo trovi in
`pipeline/data/ground_truth.json`, creato da `generator.run`).

In [ ]:
import os
from google.colab import files

os.makedirs('pipeline/data/canonical', exist_ok=True)
os.makedirs('pipeline/data', exist_ok=True)
print('Seleziona: media.csv, demand.csv, seasonality.csv, outcome.csv, ground_truth.json')
caricati = files.upload()
for nome in caricati:
    if nome == 'ground_truth.json':
        os.replace(nome, 'pipeline/data/ground_truth.json')
    elif nome.endswith('.csv'):
        os.replace(nome, os.path.join('pipeline/data/canonical', nome))

canon = sorted(f for f in os.listdir('pipeline/data/canonical') if f.endswith('.csv'))
print('CSV in canonical:', canon)
print('ground_truth presente:', os.path.exists('pipeline/data/ground_truth.json'))
attesi = {'media.csv', 'demand.csv', 'seasonality.csv', 'outcome.csv'}
assert not (attesi - set(canon)), f'Mancano CSV: {attesi - set(canon)}'
assert os.path.exists('pipeline/data/ground_truth.json'), 'Manca ground_truth.json'
print('OK: dati e verita pronti per il test.')

## 3. Fit Meridian (GPU, ~20-40 min)

Per una prova solo meccanica aggiungi `--smoke` (stime NON utilizzabili,
il recovery non sara' significativo).

In [ ]:
!python -m pipeline.model.run

## 4. Il test: parameter recovery

Confronta le stime del modello con la verita' nascosta. Cosa guardare:
- **ROI stimato vs vero** per canale;
- **copertura 90%**: quante volte il valore vero cade dentro l'intervallo
  di credibilita' del modello (piu' e' alta, meglio e');
- **MAE% sulle curve di risposta** nel range di spesa osservato (piu' e'
  basso, meglio e').

Se ROI e curve sono vicini alla verita' con buona copertura, il modello
"ci ha preso".

In [ ]:
!python -m pipeline.validation.recovery

## 5. Scarica i risultati (fit + recovery)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('dimostrazione', 'zip', 'pipeline/data/output')
files.download('dimostrazione.zip')